# Sintonia de Hiperparâmetros (Optuna) — BRKGA-SCS

Notebook dedicado para rodar a sintonia do Optuna (`brkga_scs_vr` ou `brkga_scs_vm`) e salvar os melhores parâmetros em `.json` para uso no `experiments_brkga_scs.ipynb`.

## 0. Setup — Instalação, Google Drive e Imports

In [ ]:
# Instala dependencias
!pip install -q pymoo scikit-fuzzy optuna numpy pandas
print('Dependencias instaladas.')

In [ ]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, json, importlib
from pathlib import Path

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

DATA_DIR = Path(ROOT) / 'data' / 'taillard'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_scs'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

import src.solvers.adapter
import utils.tuning_optuna
importlib.reload(src.solvers.adapter)
importlib.reload(utils.tuning_optuna)

from utils.tuning_optuna import executar_tuning_optuna, carregar_melhores_parametros

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'RESULTADOS_DIR: {RESULTADOS_DIR}')

## 1. Configuração da Sintonia

Defina abaixo o solver, `t_param`, `seeds`, `n_trials` e a lista manual de instâncias.

In [ ]:
# Escolha o solver: 'brkga_scs_vr' ou 'brkga_scs_vm'
SOLVER = 'brkga_scs_vr'

# Critério de parada por instância no tuning
# Se T_PARAM = None, usa MAX_EVALS
T_PARAM = 60
MAX_EVALS = 50000

# Quantidade de trials do Optuna e seeds por instância
N_TRIALS = 30
SEEDS = [42]
OBJECTIVE = 'Makespan'

# Passe manualmente as instâncias que deseja usar na sintonia
# Exemplo com 1 instância de diferentes níveis de Taillard:
INSTANCIAS = [
    'ta001.txt',  # 20x5
    'ta011.txt',  # 20x10
    'ta021.txt',  # 20x20
    'ta031.txt',  # 50x5
    'ta041.txt',  # 50x10
    'ta051.txt',  # 50x20
    'ta061.txt',  # 100x5
    'ta071.txt',  # 100x10
    'ta081.txt',  # 100x20
]

# Arquivo JSON onde os melhores parâmetros serão salvos
OUTPUT_JSON = RESULTADOS_DIR / f'best_params_{SOLVER}.json'

print(f'Solver: {SOLVER}')
print(f'Instâncias ({len(INSTANCIAS)}): {INSTANCIAS}')
print(f'Saída JSON: {OUTPUT_JSON}')

## 2. Executar Sintonia Optuna

In [ ]:
melhores_params = executar_tuning_optuna(
    solver=SOLVER,
    data_dir=DATA_DIR,
    output_json=OUTPUT_JSON,
    instancias_especificas=INSTANCIAS,
    n_trials=N_TRIALS,
    seeds=SEEDS,
    t_param=T_PARAM,
    max_evals=MAX_EVALS,
    objective=OBJECTIVE,
    verbose=True,
)

## 3. Conferir JSON Salvo

In [ ]:
params_carregados = carregar_melhores_parametros(OUTPUT_JSON)
print('Parâmetros prontos para usar no experiments_brkga_scs.ipynb:')
print(json.dumps(params_carregados, indent=2))